# Control quality versus prediction quality

Compare plain A-LQR, measured drift cancellation, and predicted cancellation with local or Mean Jacobian predictors as the push penalty increases. All gains use Identity dynamics and are recomputed for each penalty. **The local Jacobian is evaluated at the actual steered block input**, with the current cache held fixed. Its predicted deviation resets at each forward, so it does not propagate deviations in the KV history. Both model copies and their hidden states are float32. This experiment measures actual cached-generation states, not a full-sequence controller replay.

| Symbol / name | Meaning |
|---|---|
| $L$, $H$, $l$ | Number of blocks, hidden size, and block index $0,\ldots,L-1$. All states below are at the last position of a generation forward. |
| $Q$, $Q_F$, $R$ | State, terminal, and push-cost weights; $Q=Q_F=0.1$. Increasing $R$ generally shrinks pushes; Riccati is solved again for every $R$. Matrix costs are these scalars times identity. |
| $\lambda$, $w$ | Concept strength and drift-cancellation weight. Steer every block (`every = 1`). |
| $E_l$, $v_l$ | Cached concept vector and its unit direction, $v_l=E_l/\|E_l\|$. The cache includes the final pre-norm state, defining $v_L$. |
| $x_l$, $x_l^{\mathrm{nom}}$ | Steered and unsteered states on the same token prefix; $x_L$ and $x_L^{\mathrm{nom}}$ enter the final norm. The nominal model reads the generated tokens, not its own samples. |
| $d_l$, $\hat d_l$ | True deviation $x_l-x_l^{\mathrm{nom}}$ and predicted deviation. |
| $I$, $\Pi_l$ | Identity and off-concept projection $\Pi_l=I-v_l v_l^T$. |
| $P_l$, $u_l$ | Predictor and actual controller push at block output. Prediction uses $\hat d_0=0$, $\hat d_{l+1}=P_l\hat d_l+u_l$. |
| $a$ (`along_v`) | $v_L^T(x_L-x_L^{\mathrm{nom}})$, averaged over valid generated-token steps. |
| $o$ (`orthogonal`) | $\|\Pi_L(x_L-x_L^{\mathrm{nom}})\|/\|x_L^{\mathrm{nom}}\|$, averaged over valid steps. |
| `dev` (`change`) | $\|x_L-x_L^{\mathrm{nom}}\|/\|x_L^{\mathrm{nom}}\|$, averaged over valid steps. |
| `push_energy` | Token mean of $\sum_l\|u_l\|^2/\|x_l\|^2$, the existing normalized effort metric. |
| `none`, `meas` | Plain A-LQR ($w=0$) and measured cancellation ($w=2$). `none` is steered, not an unsteered baseline. |
| `pred_local`, `pred_mean` | Local-Jacobian and Mean-Jacobian predicted cancellation, respectively ($w=2$). All four controllers use Identity gains. |
| $f$ | $(o_{\mathrm{none}}-o_{\mathrm{pred}})/(o_{\mathrm{none}}-o_{\mathrm{meas}})$: fraction of measured drift reduction recovered. One means as good as measuring; zero means no reduction. Values are not clipped. |
| Prediction error | $\|\Pi_L(\hat d_L-d_L)\|/\|\Pi_L d_L\|$, at the final pre-norm state. |
| Step 0 / later | The prompt forward predicting the first generated token / subsequent cached forwards. |


## 1. Config and paths

For saved-CSV judging or analysis, run this cell, skip loading and generation, and start at Judge or Summary. The sweep checkpoints completed cells and resumes without generating them again. Existing caches and experiment files outside this new run directory are read-only.


In [ ]:
import gc
import json
import random
import sys
from contextlib import contextmanager
from functools import partial
from inspect import signature
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.nn.attention import SDPBackend, sdpa_kernel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'llm' / 'steering.py').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from llm.model import load_model, blocks, final_norm
from llm.steering import LQRSteering, DriftLQRSteering, sample
from llm.predicted_drift import PredictedDriftLQRSteering
from llm.trace import completion_metrics
from llm.judge import build_judge, concept_criterion, COHERENCE, repetition

MODEL_NAME = 'Qwen/Qwen3-4B'
CONCEPT = 'football'
PROMPT = 'Once upon a time'
SEED = 0
DIRECTION_SAMPLES = 200
JACOBIAN_SAMPLES = 50
MAX_CTX = 32
Q, QF = 0.1, 0.1
R_VALUES = (10, 30, 100, 300, 1000)
LAMBDAS = (2, 3, 4, 5, 6)
CONTROLLERS = ('none', 'meas', 'pred_local', 'pred_mean')
EVERY, DRIFT_WEIGHT, TRIALS = 1, 2.0, 20
MAX_NEW_TOKENS = 96
TEMPERATURE, TOP_P = 0.7, 0.95
JUDGE_BATCH = 512

ARTIFACTS = REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower()
RUN_DIR = ARTIFACTS / 'control_vs_prediction' / CONCEPT
RUN_DIR.mkdir(parents=True, exist_ok=True)
E_PATH = ARTIFACTS / f'E_{CONCEPT}_n{DIRECTION_SAMPLES}_seed{SEED}.pt'
A_PATH = ARTIFACTS / f'A_{CONCEPT}_n{JACOBIAN_SAMPLES}_ctx{MAX_CTX}_seed{SEED}.pt'
UNJUDGED_PATH = RUN_DIR / 'completions_unjudged.csv'
RESULTS_PATH = RUN_DIR / 'completions.csv'
SAMPLE_KWARGS = dict(max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P)
METRICS = ('along_v', 'orthogonal', 'change', 'push_energy')
PREDICTORS = ('local_jacobian', 'mean_jacobian')
COLORS = dict(zip(CONTROLLERS, ('tab:gray', 'tab:green', 'tab:blue', 'tab:orange')))
CONFIG = dict(model=MODEL_NAME, concept=CONCEPT, prompt=PROMPT, seed=SEED,
              direction_cache=str(E_PATH), mean_jacobian_cache=str(A_PATH),
              q=Q, qf=QF, r_values=list(R_VALUES), lambdas=list(LAMBDAS),
              controllers=list(CONTROLLERS), every=EVERY, drift_weight=DRIFT_WEIGHT,
              trials=TRIALS, dtype='float32', local_linearization='steered',
              attention_kernel='math', **SAMPLE_KWARGS, repetition_penalty=1.2)
config_path = RUN_DIR / 'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, 'Saved configuration differs; choose a new RUN_DIR.'
else:
    config_path.write_text(json.dumps(CONFIG, indent=2))
print('Results:', RUN_DIR)


## 2. FP32 models and cached operators

Math SDPA and disabled TF32 match the preceding small-deviation check. There is no BF16 fallback for either model. The existing Riccati solver retains its float64 internal arithmetic. Generation uses the unchanged sampler, including its repetition penalty of 1.2.


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
print('Free GPU GiB before loading:', torch.cuda.mem_get_info()[0] / 2**30)
tokenizer, model = load_model(MODEL_NAME, dtype=torch.float32)
nominal = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, device_map=str(model.device),
    attn_implementation=model.config._attn_implementation).eval()
model.requires_grad_(False)
nominal.requires_grad_(False)
assert all(p.dtype == torch.float32 for p in model.parameters())
assert all(p.dtype == torch.float32 for p in nominal.parameters())
NUM_LAYERS = len(blocks(model))
HIDDEN = model.get_input_embeddings().weight.shape[1]
PROMPT_LEN = len(tokenizer(PROMPT)['input_ids'])
E = torch.load(E_PATH, map_location='cpu', weights_only=True).float()
A_jac = torch.load(A_PATH, map_location='cpu', weights_only=True).to(model.device, torch.float32)
A_identity = torch.eye(HIDDEN).expand(NUM_LAYERS, -1, -1)
assert E.shape == (NUM_LAYERS + 1, HIDDEN)
assert A_jac.shape == (NUM_LAYERS, HIDDEN, HIDDEN)
print(MODEL_NAME, next(model.parameters()).dtype, next(nominal.parameters()).dtype)
print('Both FP32 models loaded. Free GPU GiB:', torch.cuda.mem_get_info()[0] / 2**30)
print('Caches:', E_PATH.name, A_PATH.name)


## 3. Observe generation without changing the controllers

The recorder delegates each push to the original controller exactly once, returns that push unchanged, and passes recorded states/commands to the existing `completion_metrics`. The original method and all observation hooks are restored on exit. Nominal forwards use the existing measured controller's cache runner; no gradients, random draws, or alternative control law are introduced.

On local-controller trajectories only, a separate Mean-Jacobian observer propagates the same commands. Its prediction never affects sampling. Both observers reset at every forward. Metrics include the step that generates EOS and exclude subsequent padding. Prediction errors are token means within each completion and phase, then completion means in the summary.


In [ ]:
class GenerationRecorder:
    def __init__(self, controller, nominal, mean_jacobian, *, predictions=False, progress=None, cell=None):
        self.controller = controller
        self.nominal = nominal
        self.mean_jacobian = mean_jacobian
        self.predictions = predictions
        self.progress, self.cell = progress, cell or {}
        self._forward_signature = signature(controller.model.base_model.forward)
        self._cache, self._x_nom = None, {}
        self.metric_steps, self.error_steps = [], []

    def _record(self, nominal, index, module, args, kwargs):
        x = (args[0] if args else kwargs['hidden_states'])[:, -1:].detach().clone()
        states = self.nominal_states if nominal else self.states
        states[index] = x
        if not nominal and index == 0:
            self.commands = {}
            if self.predictions:
                self.mean_deviation = torch.zeros_like(x)

    def _finish(self, module, args, kwargs):
        self._record(False, len(self.states) - 1, module, args, kwargs)
        assert all(x is not None for x in self.states + self.nominal_states)
        steered, reference = torch.stack(self.states), torch.stack(self.nominal_states)
        commands = self.commands
        recorded = SimpleNamespace(v=self.controller.v, steered=self.controller.steered,
                                   push=lambda layer, x: commands[layer])
        tr = dict(steered=steered, nominal=reference, controller=recorded,
                  lmbda=self.controller.lmbda,
                  valid=torch.ones(steered.shape[1:3], dtype=torch.bool, device=steered.device))
        self.metric_steps.append(completion_metrics(tr))
        if self.predictions:
            v = self.controller.v[-1].to(steered.device)
            truth = steered[-1] - reference[-1]
            truth_perp = truth - (truth @ v)[..., None] * v
            denominator = truth_perp.norm(dim=-1)
            errors = []
            for predicted in (self.controller._d, self.mean_deviation):
                error = predicted - truth
                error_perp = error - (error @ v)[..., None] * v
                errors.append(torch.where(denominator > 0, error_perp.norm(dim=-1) / denominator, torch.nan))
            self.error_steps.append(torch.stack(errors, dim=-1)[:, 0].cpu().numpy())
        if self.progress is not None:
            step = len(self.metric_steps)
            self.progress.set_postfix({**self.cell, 'step': step}, refresh=step % 16 == 0)

    @contextmanager
    def active(self, lmbda, push_from=-1):
        assert push_from == -1, 'This recorder observes last-position generation only.'
        controller = self.controller
        modules = [*blocks(controller.model), final_norm(controller.model)]
        nominal_modules = [*blocks(self.nominal), final_norm(self.nominal)]
        self.states, self.nominal_states = [None] * len(modules), [None] * len(modules)
        self.metric_steps, self.error_steps = [], []
        self._cache = None
        original_push = controller.push
        had_instance_push = 'push' in vars(controller)
        handles = []

        def record_push(layer, x):
            u = original_push(layer, x)
            self.commands[layer] = u.detach()
            if self.predictions:
                self.mean_deviation = self.mean_deviation @ self.mean_jacobian[layer].T + u
            return u

        controller.push = record_push
        try:
            with controller.active(lmbda, push_from):
                for i, module in enumerate(nominal_modules):
                    handles.append(module.register_forward_pre_hook(partial(self._record, True, i), with_kwargs=True))
                for i, module in enumerate(modules[:-1]):
                    handles.append(module.register_forward_pre_hook(partial(self._record, False, i), with_kwargs=True))
                handles.append(modules[-1].register_forward_pre_hook(self._finish, with_kwargs=True))
                if type(controller) is LQRSteering:
                    handles.append(controller.model.base_model.register_forward_pre_hook(
                        partial(DriftLQRSteering._run_nominal, self), with_kwargs=True))
                yield self
        finally:
            for handle in handles:
                handle.remove()
            if had_instance_push:
                controller.push = original_push
            else:
                del controller.push
            self._cache, self._x_nom = None, {}
            self.states, self.nominal_states, self.commands = [], [], {}
            self.mean_deviation = None

    def rows(self, ids, prompt_len):
        generated = ids[:, prompt_len:]
        eos = self.controller.model.generation_config.eos_token_id
        eos = torch.tensor(eos if isinstance(eos, list) else [eos], device=ids.device)
        is_eos = torch.isin(generated, eos).long()
        valid = ((is_eos.cumsum(1) - is_eos) == 0).cpu().numpy()
        assert generated.shape[1] == len(self.metric_steps)
        values = np.array([[[row[key] for key in METRICS] for row in step] for step in self.metric_steps])
        errors = np.stack(self.error_steps) if self.predictions else None
        rows = []
        for trial in range(ids.shape[0]):
            keep = valid[trial]
            row = {key: values[keep, trial, col].mean() for col, key in enumerate(METRICS)}
            assert all(np.isfinite(value) for value in row.values())
            row['generated_tokens'] = int(keep.sum())
            for col, predictor in enumerate(PREDICTORS):
                for phase, selection in (('step0', keep & (np.arange(len(keep)) == 0)),
                                         ('later', keep & (np.arange(len(keep)) >= 1))):
                    selected = np.array([]) if errors is None else errors[selection, trial, col]
                    finite = selected[np.isfinite(selected)]
                    row[f'{predictor}_error_{phase}'] = finite.mean() if finite.size else np.nan
            rows.append(row)
        return rows


def make_controller(name, r):
    kwargs = dict(model=model, A=A_identity, E=E, q=Q, r=r, qf=QF, every=EVERY)
    if name == 'none':
        return LQRSteering(**kwargs)
    if name == 'meas':
        return DriftLQRSteering(**kwargs, nominal=nominal, drift_weight=DRIFT_WEIGHT)
    predictor = 'local_jacobian' if name == 'pred_local' else A_jac
    return PredictedDriftLQRSteering(**kwargs, predictor=predictor, nominal=nominal, drift_weight=DRIFT_WEIGHT)


## 4. Generate and checkpoint

Twenty completions per cell. Seeds are reset to `SEED + lambda_index` before each batch, identically across controllers and penalties. Each controller is built once per penalty and reused across strengths. A failed cell is not checkpointed; rerunning resumes from complete cells. GPU OOM skips that penalty/controller's remaining cells, records the reason, and never changes precision or sampling batch size.


In [ ]:
results = pd.read_csv(UNJUDGED_PATH, keep_default_na=False).to_dict('records') if UNJUDGED_PATH.exists() else []
completed = set()
if results:
    saved = pd.DataFrame(results)
    counts = saved.groupby(['R', 'controller', 'lambda']).size()
    assert counts.eq(TRIALS).all(), 'Incomplete saved cell.'
    assert not saved.duplicated(['R', 'controller', 'lambda', 'trial']).any()
    completed = set(counts.index)
status = dict(completed_R=[], skipped_R=[], skipped_cells=[], dtype='float32', local_linearization='steered')
total = len(R_VALUES) * len(CONTROLLERS) * len(LAMBDAS)
with tqdm(total=total, initial=len(completed), desc='Control vs prediction', unit='cell') as progress:
    for r in R_VALUES:
        for name in CONTROLLERS:
            missing = [(i, strength) for i, strength in enumerate(LAMBDAS) if (r, name, strength) not in completed]
            if not missing:
                continue
            controller = recorder = ids = None
            try:
                controller = make_controller(name, r)
                for i, strength in missing:
                    cell = {'R': r, 'controller': name, 'lambda': strength}
                    recorder = GenerationRecorder(controller, nominal, A_jac, predictions=name == 'pred_local',
                                                  progress=progress, cell=cell)
                    torch.manual_seed(SEED + i)
                    with sdpa_kernel(SDPBackend.MATH):
                        ids = sample(model, tokenizer, PROMPT, TRIALS, controller=recorder, lmbda=strength, **SAMPLE_KWARGS)
                    metrics = recorder.rows(ids, PROMPT_LEN)
                    texts = tokenizer.batch_decode(ids[:, PROMPT_LEN:], skip_special_tokens=True)
                    token_rows = ids.cpu().tolist()
                    for trial, (text, values, tokens) in enumerate(zip(texts, metrics, token_rows)):
                        results.append({**cell, 'trial': trial, 'seed': SEED + i, 'every': EVERY,
                                        'drift_weight': 0.0 if name == 'none' else DRIFT_WEIGHT,
                                        **values, 'completion': text.strip(), 'token_ids': json.dumps(tokens)})
                    pd.DataFrame(results).to_csv(UNJUDGED_PATH, index=False)
                    completed.add((r, name, strength))
                    progress.update(1)
                    recorder = ids = None
            except torch.cuda.OutOfMemoryError as error:
                skipped = [dict(R=r, controller=name, **{'lambda': strength})
                           for _, strength in missing if (r, name, strength) not in completed]
                status['skipped_cells'].extend(skipped)
                print('FP32 OOM; skipping cells without changing precision:', skipped, str(error))
                progress.update(len(skipped))
            finally:
                controller = recorder = ids = None
                gc.collect()
                torch.cuda.empty_cache()
        if all((r, name, strength) in completed for name in CONTROLLERS for strength in LAMBDAS):
            status['completed_R'].append(r)
        else:
            status['skipped_R'].append(r)
        (RUN_DIR / 'run_status.json').write_text(json.dumps(status, indent=2))
print('Complete R values:', status['completed_R'], 'Incomplete/skipped:', status['skipped_R'])
print('Saved:', UNJUDGED_PATH)


## 5. Judge saved completions

Free both generation models first. Granite uses the existing batched judge and its unchanged BF16 precision; this does not affect the FP32 mechanistic measurements. Both criteria run for every saved completion, with deduplication and progress. This section is independently re-runnable from the unjudged CSV.


In [ ]:
for name in ('model', 'nominal', 'controller', 'recorder', 'A_jac', 'judge', 'ids'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
results = pd.read_csv(UNJUDGED_PATH)
results['completion'] = results['completion'].fillna('')
judge = build_judge(batch_size=JUDGE_BATCH)
texts = results['completion'].tolist()
print(f'Judging {len(texts):,} completions, {len(set(texts)):,} unique; batch size {JUDGE_BATCH}.')
results['concept'] = judge(concept_criterion(CONCEPT), PROMPT, texts, desc=f'Concept: {CONCEPT} (1/2)')
results['coherent'] = judge(COHERENCE, PROMPT, texts, desc='Coherence (2/2)')
results['both'] = results['concept'] & results['coherent']
results['repetition'] = results['completion'].map(repetition)
results.to_csv(RESULTS_PATH, index=False)
del judge
gc.collect()
torch.cuda.empty_cache()
print('Saved:', RESULTS_PATH)


## 6. Summary and prediction accuracy

Reload the judged CSV. `push_energy` is its mean; the median is separate. Deviation size is the median completion-level `change` within each strength, then the mean of these medians across strengths. Prediction errors compare both observers on **local-controller trajectories only**, using identical pushes. Later-step errors first average valid later tokens within a completion, so long completions do not receive extra weight.


In [ ]:
results = pd.read_csv(RESULTS_PATH)
keys = ['R', 'lambda', 'controller']
groups = results.groupby(keys, sort=True)
summary = groups[[*METRICS, 'concept', 'coherent', 'both', 'repetition']].mean()
summary['push_energy_median'] = groups['push_energy'].median()
summary['n'] = groups.size()
summary = summary.reset_index()
summary.to_csv(RUN_DIR / 'summary.csv', index=False)
display(summary.round(5))

deviations = (results.loc[results.controller.eq('none')].groupby(['R', 'lambda'])
              .agg(median_dev=('change', 'median'), mean_dev=('change', 'mean'), n=('change', 'size'))
              .reset_index())
deviation_size = deviations.groupby('R').median_dev.mean()
deviations['deviation_size_R'] = deviations.R.map(deviation_size)
deviations.to_csv(RUN_DIR / 'deviation_sizes.csv', index=False)

prediction_rows = []
local_rows = results.loc[results.controller.eq('pred_local')]
for (r, strength), block in local_rows.groupby(['R', 'lambda']):
    for predictor in PREDICTORS:
        for phase in ('step0', 'later'):
            values = block[f'{predictor}_error_{phase}'].dropna()
            prediction_rows.append(dict(R=r, **{'lambda': strength}, predictor=predictor, phase=phase,
                                        trajectory_controller='pred_local', mean_error=values.mean(),
                                        median_error=values.median(), error_se=values.sem(), n=len(values)))
prediction_errors = pd.DataFrame(prediction_rows)
prediction_errors.to_csv(RUN_DIR / 'prediction_errors.csv', index=False)
display(prediction_errors.round(5))


## 7. Drift reduction at matched strength and matched concept shift

Matched-strength fractions are missing when the measured reduction is below 0.02, exactly as requested. For matched concept shift, interpolate at the **first crossing along increasing strength**, rather than sorting by concept shift. Use three evenly spaced targets inside the common positive-width range. These ratios retain small nonzero denominators; each denominator is saved explicitly. A zero denominator or absent common range gives NaN. The plotted matched-shift fraction averages valid target fractions and reports their count; interpret very small denominators cautiously.


In [ ]:
def first_crossing(curve, target):
    points = curve.sort_values('lambda')[['along_v', 'orthogonal']].to_numpy(dtype=float)
    for index, (a, o) in enumerate(points):
        if not np.isfinite(a + o):
            continue
        if a == target:
            return o
        if index + 1 < len(points):
            next_a, next_o = points[index + 1]
            if np.isfinite(next_a + next_o) and next_a != a and min(a, next_a) <= target <= max(a, next_a):
                fraction = (target - a) / (next_a - a)
                return o + fraction * (next_o - o)
    return np.nan


fraction_rows = []
for (r, strength), block in summary.groupby(['R', 'lambda']):
    values = block.set_index('controller').orthogonal.reindex(CONTROLLERS)
    denominator = values['none'] - values['meas']
    for predicted in ('pred_local', 'pred_mean'):
        value = ((values['none'] - values[predicted]) / denominator
                 if np.isfinite(denominator) and denominator >= 0.02 else np.nan)
        fraction_rows.append(dict(matching='lambda', R=r, **{'lambda': strength}, controller=predicted,
                                  a_target=np.nan, o_none=values['none'], o_meas=values['meas'],
                                  o_pred=values[predicted], denominator=denominator, f=value,
                                  n_targets=int(np.isfinite(value))))

for r, block in summary.groupby('R'):
    curves = {name: block.loc[block.controller.eq(name)] for name in CONTROLLERS}
    complete = all(len(curve) == len(LAMBDAS) for curve in curves.values())
    low = max(curve.along_v.min() for curve in curves.values()) if complete else np.nan
    high = min(curve.along_v.max() for curve in curves.values()) if complete else np.nan
    targets = np.linspace(low, high, 3) if np.isfinite(low + high) and low < high else []
    for predicted in ('pred_local', 'pred_mean'):
        target_rows = []
        for target in targets:
            values = {name: first_crossing(curve, target) for name, curve in curves.items()}
            denominator = values['none'] - values['meas']
            value = ((values['none'] - values[predicted]) / denominator
                     if np.isfinite(denominator) and denominator != 0 else np.nan)
            row = dict(matching='along_v_target', R=r, **{'lambda': np.nan}, controller=predicted,
                       a_target=target, o_none=values['none'], o_meas=values['meas'],
                       o_pred=values[predicted], denominator=denominator, f=value,
                       n_targets=int(np.isfinite(value)))
            target_rows.append(row)
            fraction_rows.append(row)
        finite = [row for row in target_rows if np.isfinite(row['f'])]
        fraction_rows.append(dict(matching='along_v_mean', R=r, **{'lambda': np.nan}, controller=predicted,
                                  a_target=np.nan, o_none=np.nan, o_meas=np.nan, o_pred=np.nan,
                                  denominator=np.mean([row['denominator'] for row in finite]) if finite else np.nan,
                                  f=np.mean([row['f'] for row in finite]) if finite else np.nan,
                                  n_targets=len(finite), overlap_low=low, overlap_high=high))
f_table = pd.DataFrame(fraction_rows)
f_table['deviation_size'] = f_table.R.map(deviation_size)
f_table.to_csv(RUN_DIR / 'drift_reduction.csv', index=False)
display(f_table.loc[f_table.matching.ne('along_v_target')].round(5))


## 8. Control quality versus prediction quality

The horizontal coordinate is the no-cancellation deviation size defined above, not a predictor-specific trajectory size. Prediction error comes from the local controller's trajectories and is averaged equally over strengths. Missing matched-shift fractions remain gaps; fractions are not clipped to the unit interval.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for name in ('pred_local', 'pred_mean'):
    curve = f_table.loc[f_table.matching.eq('along_v_mean') & f_table.controller.eq(name)].sort_values('deviation_size')
    axes[0].plot(curve.deviation_size, curve.f, 'o-', color=COLORS[name], label=name)
axes[0].axhline(0, color='gray', linewidth=0.8)
axes[0].axhline(1, color='gray', linewidth=0.8, linestyle=':')
axes[0].set_ylabel('Fraction of measured drift reduction recovered')
axes[0].set_title('Matched along-concept shift')
for (predictor, phase), block in prediction_errors.groupby(['predictor', 'phase']):
    curve = block.groupby('R').mean_error.mean().reset_index()
    curve['deviation_size'] = curve.R.map(deviation_size)
    curve = curve.sort_values('deviation_size')
    name = 'pred_local' if predictor == 'local_jacobian' else 'pred_mean'
    axes[1].plot(curve.deviation_size, curve.mean_error, marker='o',
                 linestyle='-' if phase == 'step0' else '--', color=COLORS[name], label=f'{predictor}, {phase}')
axes[1].set_ylabel('Final-state rollout off-concept relative error')
axes[1].set_title('Both predictors on local-controller trajectories')
for ax in axes:
    ax.set_xscale('log')
    ticks = sorted(deviation_size.values)
    ax.set_xticks(ticks, [f'{value:.3g}' for value in ticks])
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel('No-cancellation deviation / nominal norm')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle(f'{MODEL_NAME}: {CONCEPT}, FP32, Identity gains')
fig.tight_layout()
fig.savefig(RUN_DIR / 'control_vs_prediction.png', dpi=150)
plt.show()


## 9. Judged rates at strength 4


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for name, block in summary.loc[summary['lambda'].eq(4)].groupby('controller'):
    curve = block.sort_values('R')
    for ax, metric in zip(axes, ('concept', 'coherent')):
        ax.plot(curve.R, curve[metric], 'o-', color=COLORS[name], label=name)
for ax, title in zip(axes, ('Concept rate', 'Coherence rate')):
    ax.set(xscale='log', xlabel='R (push-cost weight)', ylabel=title, ylim=(-0.05, 1.05))
    ax.grid(alpha=0.2)
axes[0].legend()
fig.suptitle(f'{CONCEPT}, lambda = 4; {TRIALS} completions per cell')
fig.tight_layout()
fig.savefig(RUN_DIR / 'judged_rates.png', dpi=150)
plt.show()


## 10. Off-concept change versus along-concept shift


In [ ]:
fig, axes = plt.subplots(1, len(R_VALUES), figsize=(4 * len(R_VALUES), 4), squeeze=False)
for ax, r in zip(axes[0], R_VALUES):
    for name, block in summary.loc[summary.R.eq(r)].groupby('controller'):
        curve = block.sort_values('lambda')
        ax.plot(curve.along_v, curve.orthogonal, 'o-', color=COLORS[name], label=name)
        for point in curve.to_dict('records'):
            ax.annotate(f'{point["lambda"]:g}', (point['along_v'], point['orthogonal']), fontsize=7,
                        xytext=(3, 3), textcoords='offset points')
    ax.set(title=f'R = {r}', xlabel='Along-concept shift (a)', ylabel='Off-concept change (o)')
    ax.grid(alpha=0.2)
axes[0, 0].legend(fontsize=8)
fig.suptitle(f'{CONCEPT}: means over {TRIALS} completions; labels = lambda')
fig.tight_layout()
fig.savefig(RUN_DIR / 'orthogonal_vs_along.png', dpi=150)
plt.show()

written = ['config.json', 'run_status.json', 'completions_unjudged.csv', 'completions.csv', 'summary.csv',
           'deviation_sizes.csv', 'drift_reduction.csv', 'prediction_errors.csv',
           'control_vs_prediction.png', 'judged_rates.png', 'orthogonal_vs_along.png']
print('Written files:')
for filename in written:
    path = RUN_DIR / filename
    assert path.is_file()
    print(path)
print('Run status:', json.loads((RUN_DIR / 'run_status.json').read_text()))
print('Local Jacobian linearization point: actual steered state, current KV cache held fixed.')
